Importing the Library

In [8]:
import pandas as pd
import os
from os import listdir
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import tensorflow
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Load the Dataset

In [9]:
path = kagglehub.dataset_download("pratik2901/multiclass-weather-dataset")

test_dir = os.path.join(path, "Multi-class Weather Dataset")

images = []
labels = []

for category in os.listdir(test_dir):
    category_path = os.path.join(test_dir, category)
    if os.path.isdir(category_path):
        for image_name in os.listdir(category_path):
            image_path = os.path.join(category_path, image_name)
            # Read the image
            image = cv2.imread(image_path)
            if image is not None:
                image = cv2.resize(image, (112, 112))
                images.append(image)
                labels.append(category)

print(f"Loaded {len(images)} images from the testing folder.")

100%|██████████| 91.4M/91.4M [00:11<00:00, 8.41MB/s]

Extracting files...


Loaded 1123 images from the testing folder.


Preprocessing (Normalization) of the images

In [10]:
preprocess_images = []

for image in images:
    # Convert BGR to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Split channels
    b, g, r = cv2.split(image_rgb)

    # Normalize each channel to [0, 1]
    b_normalized = cv2.normalize(b.astype('float32'), None, 0, 1, cv2.NORM_MINMAX)
    g_normalized = cv2.normalize(g.astype('float32'), None, 0, 1, cv2.NORM_MINMAX)
    r_normalized = cv2.normalize(r.astype('float32'), None, 0, 1, cv2.NORM_MINMAX)

    # Merge normalized channels back into a float32 RGB image
    normalized_image = cv2.merge((b_normalized, g_normalized, r_normalized))

    # Convert normalized image back to 8-bit for histogram equalization
    normalized_image_8bit = (normalized_image * 255).astype('uint8')

    # Histogram equalization on the Y channel
    img_yuv = cv2.cvtColor(normalized_image_8bit, cv2.COLOR_RGB2YUV)
    img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
    img_eq = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

    # Denoising
    dst = cv2.fastNlMeansDenoisingColored(img_eq, None, 10, 10, 7, 15)
    preprocess_images.append(dst)


Features Extraction

In [11]:
inputs = keras.Input(shape=(112, 112, 3))
x = layers.Conv2D(32, 5, strides=2, activation="relu")(inputs)
x = layers.Conv2D(32, 3, activation="relu")(x)
x = layers.Conv2D(32, 3, activation="relu")(x)
initial_model = keras.Model(inputs=inputs, outputs=x)

feature_extractor = initial_model
all_features = []

for img in preprocess_images:
    img_batch = np.expand_dims(img, axis=0)
    features = feature_extractor.predict(img_batch)
    all_features.append(features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━

Classification

In [12]:
flattened_features = []

for feature_set in all_features:
    flattened = [f.reshape(-1) for f in feature_set]
    concatenated = np.concatenate(flattened)
    flattened_features.append(concatenated)

X = np.stack(flattened_features)

X_mean = np.mean(X, axis=0)
X_centered = X - X_mean

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
X_whitened = np.dot(U, Vt)

def g(u):
    return np.tanh(u)

def g_prime(u):
    return 1 - np.tanh(u)**2

def fast_ica(X, n_components, max_iter=200, tol=1e-5):
    n_samples, n_features = X.shape
    W = np.random.randn(n_components, n_features)

    for i in range(n_components):
        w = W[i]
        for _ in range(max_iter):
            wx = np.dot(X, w)
            gwx = g(wx)
            g_wx = g_prime(wx)
            w_new = (X.T @ gwx - g_wx.mean() * w) / n_samples

            if i > 0:
                w_new -= np.dot(np.dot(W[:i], w_new), W[:i])

            w_new /= np.linalg.norm(w_new)
            if np.abs(np.abs(np.dot(w, w_new)) - 1) < tol:
                break
            w = w_new
        W[i] = w
    S = np.dot(X, W.T)
    return S, W

X_ica, mixing_matrix = fast_ica(X, n_components=50)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X_ica, labels, test_size=0.2, random_state=42)

svm = SVC(kernel='linear', C=1.0)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

Evalution

In [14]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7333333333333333
Classification Report:
               precision    recall  f1-score   support

      Cloudy       0.59      0.69      0.64        61
        Rain       0.60      0.57      0.58        44
       Shine       0.76      0.74      0.75        53
     Sunrise       0.97      0.88      0.92        67

    accuracy                           0.73       225
   macro avg       0.73      0.72      0.72       225
weighted avg       0.74      0.73      0.74       225

